In [3]:
import pandas as pd
from scipy.stats import stats


## 1.Read CSV Files

In [6]:
Fact_Sales = pd.read_csv(r"C:\Users\rebuytech\Downloads\Compressed\walmart-recruiting-store-sales-forecasting\Fact_Sales.csv")
Dim_Stores = pd.read_csv(r"C:\Users\rebuytech\Downloads\Compressed\walmart-recruiting-store-sales-forecasting\Dim_Stores.csv")
Dim_Features = pd.read_csv(r"C:\Users\rebuytech\Downloads\Compressed\walmart-recruiting-store-sales-forecasting\Dim_Features.csv")

## 2. Merge Data 

In [7]:
merged_df = pd.merge(Fact_Sales,Dim_stores , on="Store" , how="inner")
final_df = pd.merge(merged_df,Dim_Features , on=["Store","Date"] , how="inner")

In [9]:
final_df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Is_Return,Is_Zero_Sales,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y
0,1,1,2010-02-05,24924.50,False,False,False,A,151315,42.31,2.572,0.0,0.0,0.0,0.0,0.0,211.09636,8.106,False
1,1,1,2010-02-12,46039.49,True,False,False,A,151315,38.51,2.548,0.0,0.0,0.0,0.0,0.0,211.24217,8.106,True
2,1,1,2010-02-19,41595.55,False,False,False,A,151315,39.93,2.514,0.0,0.0,0.0,0.0,0.0,211.28914,8.106,False
3,1,1,2010-02-26,19403.54,False,False,False,A,151315,46.63,2.561,0.0,0.0,0.0,0.0,0.0,211.31964,8.106,False
4,1,1,2010-03-05,21827.90,False,False,False,A,151315,46.50,2.625,0.0,0.0,0.0,0.0,0.0,211.35014,8.106,False


## 3.Data Spilliting (TypeA)

In [10]:
df_type_a = final_df[final_df['Type'] == "A"]

In [12]:
df_type_a.tail()

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Is_Return,Is_Zero_Sales,Type,Size,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y
391055,41,99,2012-09-14,0.10,False,False,False,A,196321,59.39,3.659,10525.10,0.00,11.14,1373.26,6742.72,198.12672,6.432,False
391056,41,99,2012-09-21,0.08,False,False,False,A,196321,59.81,3.765,11062.12,72.64,6.00,1125.21,7962.34,198.35852,6.432,False
391057,41,99,2012-10-05,934.88,False,False,False,A,196321,50.14,3.779,6094.23,0.00,33.94,2887.65,3853.33,198.82213,6.195,False
391058,41,99,2012-10-12,230.03,False,False,False,A,196321,39.38,3.760,1570.23,0.00,26.31,834.80,14421.12,199.05394,6.195,False
391059,41,99,2012-10-26,29.88,False,False,False,A,196321,41.80,3.686,4864.30,101.34,250.60,47.24,1524.43,199.21953,6.195,False


In [14]:
len(df_type_a)

215478

## 4.Create Our Testing Groups

In [18]:
# Split Type A sales into control and treatment groups based on MarkDown1
control_group = df_type_a[df_type_a['MarkDown1'] == 0]['Weekly_Sales']  # No markdown promotion
variant_group = df_type_a[df_type_a['MarkDown1'] > 0]['Weekly_Sales']   # Markdown promotion present

In [20]:
print("Group A len:", len(control_group))
print("Group B len:", len(variant_group))

Group A len: 138523
Group B len: 76955


## 5.Calculate Mean For Each Group

In [26]:
mean_control = control_group.mean()
mean_variant = variant_group.mean()
print(f"Control Group mean: {mean_control:.2f}")
print(f"Variant Group mean: {mean_variant:.2f}")

Control Group mean: 19907.31
Variant Group mean: 20445.64


In [29]:
lift_percentage = ((mean_variant - mean_control) / mean_control) * 100
print(f"lift percentage : {lift_percentage: .2f}%")

lift percentage :  2.70%


## 7. Make T-Test & calculate P-value

In [30]:
t_stat, p_value = stats.ttest_ind(variant_group, control_group, equal_var=False)
print("--- T-test Result---")
print(f"T-Statistic: {t_stat:.2f}")
print(f"P-Value: {p_value:.5f}")

--- T-test Result---
T-Statistic: 4.48
P-Value: 0.00001


C:\Users\rebuytech\AppData\Local\Temp\ipykernel_24444\3700491426.py:1: DeprecationWarning: Please import `ttest_ind` from the `scipy.stats` namespace; the `scipy.stats.stats` namespace is deprecated and will be removed in SciPy 2.0.0.
  t_stat, p_value = stats.ttest_ind(variant_group, control_group, equal_var=False)


In [ ]:
if p_value < 0.05:
    if mean_variant > mean_control:
        print("Decision: The discount was successful! The increase is real and statistically significant, and the discount can be generalized.")
    else:
        print("Decision: The discount is hurting us! Sales declined significantly in a statistically significant way.")
else:
    print("Decision: There is no real difference. The change in sales is due to chance or normal fluctuation, and the discount was not effective.")

القرار: الخصم ناجح! الزيادة حقيقية وذات دلالة إحصائية، ويمكن تعميم الخصم.
